# Section B - SQL Engineering


## Q1. Intermediate SQL Queries


In [1]:
import pandas as pd
import sqlite3

In [2]:
SCHEMA = """
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS clickstream;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT,
    email         TEXT,
    city          TEXT,
    signup_date   TEXT
);

CREATE TABLE products (
    product_id    TEXT PRIMARY KEY,
    product_name  TEXT,
    category      TEXT,
    supplier_id   TEXT,
    cost_price    REAL,
    selling_price REAL
);

CREATE TABLE orders (
    order_id       TEXT PRIMARY KEY,
    customer_id    TEXT REFERENCES customers(customer_id),
    product_id     TEXT REFERENCES products(product_id),
    order_date     TEXT,
    quantity       INTEGER,
    unit_price     REAL,
    payment_status TEXT
);

CREATE TABLE clickstream (
    event_id        TEXT,
    customer_id     TEXT, --REFERENCES customers(customer_id),
    event_type      TEXT,
    page_url        TEXT,
    event_timestamp TEXT,
    device_type     TEXT
);
"""

# Indexes are created *after* loading -- see note at the bottom of the file.
INDEXES = """
CREATE INDEX idx_orders_customer   ON orders(customer_id);
CREATE INDEX idx_orders_product    ON orders(product_id);
CREATE INDEX idx_orders_date       ON orders(order_date);
CREATE INDEX idx_click_customer_ts ON clickstream(customer_id, event_timestamp);
CREATE INDEX idx_click_type        ON clickstream(event_type);
"""

db = 'quick_cart.db'
conn = sqlite3.connect(db)
conn.executescript(SCHEMA)

In [3]:
customers_df = pd.read_csv('/content/drive/MyDrive/Data/ironhack/Mini project/files/customers.csv')
customers_df.to_sql('customers', conn, if_exists='append', index=False)


products_df = pd.read_csv('/content/drive/MyDrive/Data/ironhack/Mini project/files/products.csv')
products_df.to_sql('products', conn, if_exists='append', index=False)


orders_df = pd.read_csv('/content/drive/MyDrive/Data/ironhack/Mini project/files/orders.csv')
orders_df.to_sql('orders', conn, if_exists='append', index=False)


clickstream_df = pd.read_csv('/content/drive/MyDrive/Data/ironhack/Mini project/files/clickstream.csv')
clickstream_df.to_sql('clickstream', conn, if_exists='append', index=False)
clickstream_df

,event_id,customer_id,event_type,page_url,event_timestamp,device_type
0,1861,7.0,purchase,/help,2025-02-05 08:29:38,Tablet
1,354,83.0,Page_View,/product/101,2025-07-13 20:51:00,MOBILE
2,1334,97.0,CLICK,/account,03/03/2025 18:50,NaN
3,906,91.0,ADD TO CART,/orders,04/06/2025 06:10,MOBILE
4,1290,25.0,page_view,/orders,2025-01-15 20:47:38,MOBILE
...,...,...,...,...,...,...
1995,1131,6.0,Page_View,/product/105,2025-01-02T11:44:05Z,Tablet
1996,1295,66.0,ADD TO CART,/home,2025-02-13 18:24:46,NaN
1997,861,53.0,Search,/help,2025-03-15 10:20:07,tablet
1998,1460,6.0,page_view,/products,2025-07-06T01:14:53Z,NaN


In [4]:
query = """
SELECT * FROM customers LIMIT 5;
-- SELECT * FROM products LIMIT 5;
-- SELECT * FROM orders LIMIT 5;
-- SELECT * FROM clickstream LIMIT 5;
"""
pd.read_sql(query, conn)

,customer_id,customer_name,email,city,signup_date
0,1,William Jennings,william.jennings1@example.com,Dublin,2025-01-29
1,2,Diane Newman,diane.newman2@example.com,Manchester,2025-11-06
2,3,Samuel Wright,samuel.wright3@example.com,London,2024-08-12
3,4,Gillian Barnes,gillian.barnes4@example.com,Lisbon,2025-09-15
4,5,Nigel Edwards,nigel.edwards5@example.com,Glasgow,2025-11-23


### 1. Find top 10 customers by revenue.  

In [5]:
query = """
SELECT c.customer_name, o.unit_price*o.quantity as revenue
FROM  customers c LEFT JOIN orders o
ON c.customer_id = o.customer_id
WHERE LOWER(o.payment_status) = 'paid'
GROUP BY c.customer_name
ORDER BY revenue DESC
LIMIT 10
"""
df = pd.read_sql(query,conn)
df

,customer_name,revenue
0,Mr Russell Jennings,5997.0
1,Susan Shepherd,4998.0
2,Fiona Saunders-Mitchell,4998.0
3,Eric O'Neill,4998.0
4,Mrs Lorraine Sutton,3998.0
5,Charlie Ward,3998.0
6,Carole Hussain-Booth,3196.0
7,Ms Elaine Taylor,2598.0
8,Sylvia Kelly-Ward,2499.0
9,Nicole Baldwin,2499.0


### 2. Find month-over-month sales growth.

In [6]:
query = """
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date) AS month,
           SUM(quantity * unit_price)    AS revenue
    FROM orders
    WHERE LOWER(payment_status) = 'paid'
    GROUP BY 1
)
SELECT month,
       revenue AS revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month,
       revenue - LAG(revenue) OVER (ORDER BY month) AS abs_change,
       100 * (revenue - LAG(revenue) OVER (ORDER BY month))/ LAG(revenue) OVER (ORDER BY month) AS pct_growth
FROM monthly
ORDER BY month;
"""
df = pd.read_sql(query,conn)
df

,month,revenue,prev_month,abs_change,pct_growth
0,2024-09,10694.0,NaN,NaN,NaN
1,2024-10,10794.0,10694.0,100.0,0.935104
2,2024-11,3998.0,10794.0,-6796.0,-62.960904
3,2024-12,745.0,3998.0,-3253.0,-81.365683
4,2025-01,15843.0,745.0,15098.0,2026.577181
5,2025-02,12238.0,15843.0,-3605.0,-22.754529
6,2025-03,7952.0,12238.0,-4286.0,-35.022062
7,2025-04,8136.0,7952.0,184.0,2.313883
8,2025-05,26485.0,8136.0,18349.0,225.528515
9,2025-06,8410.0,26485.0,-18075.0,-68.246177


### 3. Find customers who ordered in consecutive months.  

In [7]:
query = """
WITH customer_months AS (
    SELECT DISTINCT
           customer_id,
           CAST(strftime('%Y', order_date) AS INTEGER) * 12 + CAST(strftime('%m', order_date) AS INTEGER) AS month_idx,
           strftime('%Y-%m', order_date) AS month
    FROM orders
    WHERE LOWER(payment_status) = 'paid'
),
flagged AS (
    SELECT customer_id, month, month_idx,
           LAG(month_idx) OVER (PARTITION BY customer_id ORDER BY month_idx) AS prev_idx
    FROM customer_months
)
SELECT c.customer_id, c.customer_name, COUNT(*) AS consecutive_orders
FROM flagged f
JOIN customers c ON c.customer_id = f.customer_id
WHERE f.month_idx - f.prev_idx = 1
GROUP BY 1, 2
ORDER BY consecutive_orders DESC;
"""
df = pd.read_sql(query,conn)
df

,customer_id,customer_name,consecutive_orders
0,74,Mrs Lesley Charlton,3
1,95,Miss Olivia Taylor,3
2,65,Mr Matthew Hunt,2
3,69,Miss Danielle Martin,2
4,79,Sylvia Kelly-Ward,2
5,1,William Jennings,1
6,10,Wayne Gough,1
7,12,Sean Norton,1
8,16,Dr Jonathan Middleton,1
9,17,Diana Cross-Miah,1


### 4. Find products never ordered.

In [8]:
q = """
SELECT p.product_id, p.product_name, p.selling_price
FROM products p
LEFT JOIN orders o ON o.product_id = p.product_id
WHERE o.product_id IS NULL
"""
df = pd.read_sql(q,conn)

df

,product_id,product_name,selling_price


### 5. Find revenue contribution percentage by category.  

In [9]:

q = """
WITH cat AS (SELECT p.category, SUM(o.unit_price*o.quantity) AS sum_sales
FROM products p
LEFT JOIN orders o ON o.product_id = p.product_id
WHERE LOWER(o.payment_status) = 'paid'
GROUP BY p.category
)

SELECT category, sum_sales, ROUND((sum_sales/SUM(sum_sales) OVER())*100,2) AS percentage
FROM cat
order by percentage desc
"""
df = pd.read_sql(q,conn)

df

,category,sum_sales,percentage
0,Furniture,466504.0,45.97
1,Electronics,447724.0,44.12
2,Accessories,72355.0,7.13
3,Home & Living,17135.0,1.69
4,Stationery,11125.0,1.10


## Q2. Advanced SQL


### 1. Rank customers based on total revenue.

In [10]:
query = """
SELECT c.customer_name, SUM(o.unit_price*o.quantity) as total_revenue
FROM  customers c LEFT JOIN orders o
ON c.customer_id = o.customer_id
WHERE LOWER(o.payment_status) = 'paid'
GROUP BY c.customer_id
ORDER BY total_revenue DESC
"""
df = pd.read_sql(query,conn)
df

,customer_name,total_revenue
0,Miss Olivia Taylor,42015.0
1,Mr Russell Jennings,28484.0
2,Mrs Jessica Jackson,27782.0
3,Mr Matthew Hunt,27629.0
4,Mrs Lesley Charlton,27622.0
...,...,...
93,Christine Taylor,698.0
94,Dr Trevor Tomlinson,267.0
95,Malcolm Coates,178.0
96,Tracy Harrison,178.0


### 2. Find running total sales by month.


In [11]:
q = """
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date)  AS month,
           SUM(quantity * unit_price)     AS sales
    FROM orders
    WHERE LOWER(payment_status) = 'paid'
    GROUP BY 1
)
SELECT month,
       sales,
       SUM(sales) OVER (ORDER BY month) AS running_total_sales
FROM monthly
ORDER BY month;
"""
df = pd.read_sql(q,conn)
df

,month,sales,running_total_sales
0,2024-09,10694.0,10694.0
1,2024-10,10794.0,21488.0
2,2024-11,3998.0,25486.0
3,2024-12,745.0,26231.0
4,2025-01,15843.0,42074.0
5,2025-02,12238.0,54312.0
6,2025-03,7952.0,62264.0
7,2025-04,8136.0,70400.0
8,2025-05,26485.0,96885.0
9,2025-06,8410.0,105295.0


### 3. Find the highest selling product per category.  

In [12]:
q = """
WITH product_sales AS (
    SELECT p.category,
           p.product_id,
           p.product_name,
           SUM(o.quantity * o.unit_price) AS revenue
    FROM orders o
    JOIN products p ON p.product_id = o.product_id
    WHERE LOWER(o.payment_status) = 'paid'
    GROUP BY p.category, p.product_id, p.product_name
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rn
    FROM product_sales
)
SELECT category, product_id, product_name, revenue
FROM ranked
WHERE rn = 1
ORDER BY revenue DESC;
"""
df = pd.read_sql(q,conn)
df

,category,product_id,product_name,revenue
0,Electronics,105,4K Monitor,293853.0
1,Furniture,109,Standing Desk,274890.0
2,Accessories,110,Backpack,72355.0
3,Home & Living,108,Water Bottle,17135.0
4,Stationery,104,Notebook Pack,11125.0


### 4. Find 7-day rolling average sales.

In [13]:
q = """
WITH daily AS (
    SELECT DATE(order_date) AS day,
           SUM(quantity * unit_price) AS sales
    FROM orders
    WHERE LOWER(payment_status) = 'paid'
    GROUP BY 1
)
SELECT day,
       sales,
       AVG(sales) OVER (ORDER BY day ROWS 6 PRECEDING) AS rolling_7d_avg,
       SUM(sales) OVER (ORDER BY day ROWS 6 PRECEDING) AS rolling_7d_total
FROM daily
ORDER BY day;
"""
df = pd.read_sql(q,conn)
df

,day,sales,rolling_7d_avg,rolling_7d_total
0,2024-09-26,698.0,698.000000,698.0
1,2024-09-30,9996.0,5347.000000,10694.0
2,2024-10-06,9995.0,6896.333333,20689.0
3,2024-10-15,799.0,5372.000000,21488.0
4,2024-11-05,3998.0,5097.200000,25486.0
...,...,...,...,...
308,2028-02-08,356.0,1241.285714,8689.0
309,2028-03-03,998.0,1170.000000,8190.0
310,2028-03-26,2598.0,899.571429,6297.0
311,2028-03-29,1999.0,1135.285714,7947.0


# Section C - Python + Pandas Engineering

## Q1. Data Ingestion Pipeline

### Task 1. Read datasets.

In [14]:
# Task 1. Read datasets.
customers = pd.read_sql('SELECT * FROM customers;', conn)
products = pd.read_sql('SELECT * FROM products;', conn)
orders = pd.read_sql('SELECT * FROM orders;', conn)
clickstream = pd.read_sql('SELECT * FROM clickstream;', conn)

### Task 2. Validate schema consistency.

In [15]:
## CUSTOMERS
customers.customer_id = customers.customer_id.astype(int)
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['email'] = customers['email'].str.strip().str.lower()
customers["email"] = customers["email"].str.replace(r"\d", "", regex=True)

print('===== CUSTOMER SCHEMA ======')
print(customers.info())

## PRODUCTS
products['product_id'] = products['product_id'].astype(int)
products['supplier_id'] = products['supplier_id'].astype(int)

print('\n===== PRODUCTS SCHEMA ======')
print(products.info())


## ORDERS
orders['order_id'] = orders['order_id'].astype(int)
orders['customer_id'] = orders['customer_id'].astype(int)
orders['product_id'] = orders['product_id'].astype(int)
orders['order_date'] = pd.to_datetime(orders['order_date'])

print('\n===== ORDERS SCHEMA ======')
print(orders.info())

## CLICKSTREAM
print('\n===== CLICKSTREAM SCHEMA ======')
clickstream['customer_id'] = clickstream['customer_id'].astype(float).astype('Int64', errors = 'ignore')
clickstream['device_type'] = clickstream['device_type'].str.lower()
clickstream['event_timestamp'] = pd.to_datetime(clickstream['event_timestamp'], format='mixed', utc=True)
clickstream['event_type'] = clickstream['event_type'].apply(lambda x: x.lower().strip().replace('_', '').replace('-', '').replace(' ', '').replace('serach','search').replace('clik', 'click'))

print(clickstream.info())
# print(clickstream.info())



# Task 4. Identify invalid records.


===== CUSTOMER SCHEMA ======
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    100 non-null    int64         
 1   customer_name  100 non-null    object        
 2   email          100 non-null    object        
 3   city           100 non-null    object        
 4   signup_date    100 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 4.0+ KB
None

===== PRODUCTS SCHEMA ======
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     10 non-null     int64  
 1   product_name   10 non-null     object 
 2   category       10 non-null     object 
 3   supplier_id    10 non-null     int64  
 4   cost_price     10 non-null     float6

### Task 2. Validate schema consistency.

In [ ]:
## CUSTOMERS
customers.customer_id = customers.customer_id.astype(int)
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['email'] = customers['email'].str.strip().str.lower()
customers["email"] = customers["email"].str.replace(r"\d", "", regex=True)

print('===== CUSTOMER SCHEMA ======')
print(customers.info())

## PRODUCTS
products['product_id'] = products['product_id'].astype(int)
products['supplier_id'] = products['supplier_id'].astype(int)

print('\n===== PRODUCTS SCHEMA ======')
print(products.info())


## ORDERS
orders['order_id'] = orders['order_id'].astype(int)
orders['customer_id'] = orders['customer_id'].astype(int)
orders['product_id'] = orders['product_id'].astype(int)
orders['order_date'] = pd.to_datetime(orders['order_date'])

print('\n===== ORDERS SCHEMA ======')
print(orders.info())

## CLICKSTREAM
print('\n===== CLICKSTREAM SCHEMA ======')
clickstream['customer_id'] = clickstream['customer_id'].astype(float).astype('Int64', errors = 'ignore')
clickstream['device_type'] = clickstream['device_type'].str.lower()
clickstream['event_timestamp'] = pd.to_datetime(clickstream['event_timestamp'], format='mixed', utc=True)
clickstream['event_type'] = clickstream['event_type'].apply(lambda x: x.lower().strip().replace('_', '').replace('-', '').replace(' ', '').replace('serach','search').replace('clik', 'click'))

print(clickstream.info())
# print(clickstream.info())



# Task 4. Identify invalid records.


===== CUSTOMER SCHEMA ======
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    100 non-null    int64         
 1   customer_name  100 non-null    object        
 2   email          100 non-null    object        
 3   city           100 non-null    object        
 4   signup_date    100 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 4.0+ KB
None

===== PRODUCTS SCHEMA ======
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     10 non-null     int64  
 1   product_name   10 non-null     object 
 2   category       10 non-null     object 
 3   supplier_id    10 non-null     int64  
 4   cost_price     10 non-null     float6

## Q3. Clickstream Analytics

### 1. Find the most visited pages.  

In [18]:
clickstream.page_url.value_counts().reset_index()

,page_url,count
0,/products,212
1,/home,204
2,/search,203
3,/product/105,203
4,/account,202
5,/orders,188
6,/product/101,187
7,/cart,176
8,/checkout,175
9,/help,167


### 2. Calculate session counts.  

In [26]:
(
    clickstream
    .groupby("customer_id")
    .count().event_id.reset_index()
    .rename(columns = {"event_id":"session_count"})
    .sort_values("session_count", ascending = False, ignore_index=True)
    )

,customer_id,session_count
0,85,33
1,36,28
2,6,28
3,16,27
4,53,27
...,...,...
95,7,11
96,40,11
97,25,11
98,71,11


### 3.  Find bounce rate.  

In [42]:
(
    clickstream
    # .query("event_id == '2'")
    # .customer_id.value_counts()
    )

,event_id,customer_id,event_type,page_url,event_timestamp,device_type
0,1861,7,purchase,/help,2025-02-05 08:29:38+00:00,tablet
1,354,83,pageview,/product/101,2025-07-13 20:51:00+00:00,mobile
2,1334,97,click,/account,2025-03-03 18:50:00+00:00,None
3,906,91,addtocart,/orders,2025-04-06 06:10:00+00:00,mobile
4,1290,25,pageview,/orders,2025-01-15 20:47:38+00:00,mobile
...,...,...,...,...,...,...
1995,1131,6,pageview,/product/105,2025-01-02 11:44:05+00:00,tablet
1996,1295,66,addtocart,/home,2025-02-13 18:24:46+00:00,None
1997,861,53,search,/help,2025-03-15 10:20:07+00:00,tablet
1998,1460,6,pageview,/products,2025-07-06 01:14:53+00:00,None


### 4. Find mobile vs desktop traffic percentage.

In [36]:
total = len(clickstream)
(
    clickstream
    .device_type
    .value_counts()
    .apply(lambda x: round(100*(x/total),2))
    .reset_index()
    .rename(columns = {"count":"pct_ratio",})
    )

,device_type,pct_ratio
0,mobile,33.45
1,desktop,22.70
2,tablet,22.65


## Q4. Export Optimization

In [56]:
import time
t = time.time()
clickstream.to_csv('clickstream.csv', index=False)
print(f"csv exported in {(time.time()-t):.3f} seconds")
t = time.time()
clickstream.to_parquet('clickstream.parquet', index=False)
print(f"parquet exported in {(time.time()-t):.3f} seconds")

t = time.time()
pd.read_csv('clickstream.csv')
# check csv size

print(f"csv read in {(time.time()-t):.3f} seconds")

t = time.time()
pd.read_parquet('clickstream.parquet')
print(f"parquet read in {(time.time()-t):.3f} seconds")

import os

csv_file_size = os.path.getsize('clickstream.csv')
parquet_file_size = os.path.getsize('clickstream.parquet')

print(f"Size of clickstream.csv: {int(csv_file_size/1000)} kb")
print(f"Size of clickstream.parquet: {int(parquet_file_size/1000)} kb")

csv exported in 0.014 seconds
parquet exported in 0.004 seconds
csv read in 0.003 seconds
parquet read in 0.003 seconds
Size of clickstream.csv: 111 kb
Size of clickstream.parquet: 38 kb
